# Lab 13 — Part 2: Search Functions and Retrieval Approaches

**Learning goals:** Reuse the retrieval pipeline in a function, explore different queries, and compare sparse and dense retrieval.

## Setup — Run this cell first

The cell below recreates the small document collection and TF-IDF vectors used in Part 1.

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

documents = [
    "Transformers use self-attention to understand relationships between words.",
    "TF-IDF represents documents using word importance scores.",
    "Word2Vec learns dense vector representations of words.",
    "Named entity recognition identifies names, places, and organizations in text.",
    "Information retrieval finds relevant documents for a user query.",
    "Cosine similarity measures how similar two text vectors are."
]

vectorizer = TfidfVectorizer(stop_words="english")
document_vectors = vectorizer.fit_transform(documents)

## 1. Make a simple search function

Now combine the steps so you can search the document collection using any query.

In [2]:
def search(query, k=2):
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, document_vectors)[0]
    top_ids = scores.argsort()[::-1][:k]

    for i in top_ids:
        print(f"Score: {scores[i]:.3f} | {documents[i]}")

search("How can we measure similarity between text vectors?")

Score: 0.686 | Cosine similarity measures how similar two text vectors are.
Score: 0.148 | Named entity recognition identifies names, places, and organizations in text.


## 2. Sparse vs. dense retrieval (conceptual comparison)

| Sparse retrieval | Dense retrieval |
|---|---|
| Represents text using word-based features, e.g., TF-IDF | Represents text using learned embeddings |
| Works well when important query words appear in documents | Can identify similar meanings even when different words are used |
| Simple and lightweight | Usually requires a pretrained embedding model |

**Important limitation:** TF-IDF mainly matches words, not meanings. A query about *"automobiles"* may not retrieve a document about *"cars"* unless the words also overlap in other ways.

In modern RAG, retrieval can be sparse, dense, or a combination of both.

## 3. Does a high-ranked result always mean a good match?

The search function returns the **highest-scoring** documents, even when all the scores are zero. A high *rank* does not always mean a high *similarity score*.

For example, our collection is about NLP. What happens when we search for an unrelated topic like `"volcano eruption"`?

In [3]:
search("volcano eruption", k=2)

# Compare it with a relevant query
print("--- Relevant query ---")
search("cosine similarity", k=2)

Score: 0.000 | Cosine similarity measures how similar two text vectors are.
Score: 0.000 | Information retrieval finds relevant documents for a user query.
--- Relevant query ---
Score: 0.594 | Cosine similarity measures how similar two text vectors are.
Score: 0.000 | Information retrieval finds relevant documents for a user query.


## 4. Improve search: skip results with a score of zero

We can improve the function by printing only documents with a **positive similarity score**.

In [4]:
def search_filtered(query, k=2):
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, document_vectors)[0]
    top_ids = scores.argsort()[::-1][:k]

    found = False
    for i in top_ids:
        if scores[i] > 0:
            print(f"Score: {scores[i]:.3f} | {documents[i]}")
            found = True
    if not found:
        print("No matching documents found.")

search_filtered("volcano eruption")
print("---")
search_filtered("cosine similarity")

No matching documents found.
---
Score: 0.594 | Cosine similarity measures how similar two text vectors are.


## 5. Practice questions

1. Use `search()` to find a note about **named entity recognition**. Which document ranks first?
2. Add `"BERT is a transformer model that uses bidirectional context."` to `documents`, rebuild the TF-IDF vectors using `vectorizer.fit_transform(documents)`, and search for `"transformer models"`. Why was rebuilding necessary?

In [5]:
# Practice question 1: search for named entity recognition
search("named entity recognition", k=2)

Score: 0.625 | Named entity recognition identifies names, places, and organizations in text.
Score: 0.000 | Cosine similarity measures how similar two text vectors are.


### Solution

#### Question 1 — Which document ranks first for *named entity recognition*?

**Answer:** The first result is *"Named entity recognition identifies names, places, and organizations in text."* with a score of about **0.625**. It is the only document that contains the words *named*, *entity* and *recognition*.

The second result has a score of **0.000**: `search()` always returns `k` documents, even when they do not match (as seen in section 3). `search_filtered()` shows only the real match:

In [6]:
search_filtered("named entity recognition", k=2)

Score: 0.625 | Named entity recognition identifies names, places, and organizations in text.


#### Question 2 — Add a new document about BERT and search for *transformer models*

**Step 1 — Add the document, but do not rebuild yet.** What happens if we search with the old vectors?

In [7]:
documents.append("BERT is a transformer model that uses bidirectional context.")

print("Number of documents:", len(documents))
print("Rows in the old TF-IDF matrix:", document_vectors.shape[0])
print("'transformer' in old vocabulary:", "transformer" in vectorizer.vocabulary_)
print()
search_filtered("transformer models")

Number of documents: 7
Rows in the old TF-IDF matrix: 6
'transformer' in old vocabulary: False

No matching documents found.


Nothing is found. The new document is in the `documents` list, but it has **no row** in the old `document_vectors` matrix, and the words *transformer* and *models* are not in the old vocabulary, so the query vector is all zeros.

**Step 2 — Rebuild the TF-IDF vectors and search again.**

In [8]:
document_vectors = vectorizer.fit_transform(documents)

print("New TF-IDF matrix shape:", document_vectors.shape)
print("'transformer' in new vocabulary:", "transformer" in vectorizer.vocabulary_)
print()
search("transformer models", k=2)

New TF-IDF matrix shape: (7, 45)
'transformer' in new vocabulary: True

Score: 0.408 | BERT is a transformer model that uses bidirectional context.
Score: 0.000 | Cosine similarity measures how similar two text vectors are.


**Result:** after rebuilding, the matrix grows from **6 × 39** to **7 × 45** (6 new words: *bert*, *bidirectional*, *context*, *model*, *transformer*, *uses*), and the new BERT document is ranked first with a score of about **0.408**.

**Why was rebuilding necessary?**

- `fit_transform()` **learns the vocabulary and the IDF weights** from the documents it receives and then creates their vectors. The vectorizer and `document_vectors` were built from the original 6 documents, so they knew nothing about the new one.
- Appending to the `documents` list does **not** update the vectors: without rebuilding, the new document has no vector, so it can never be compared with a query or retrieved.
- New words (*transformer*, *model*, *BERT*, ...) are not in the old vocabulary. `transform()` **ignores unknown words**, so the query *"transformer models"* became an all-zero vector.
- **IDF depends on the whole collection** (the number of documents and how many contain each word). Adding a document changes the IDF values, so all document vectors should be recomputed for the scores to be correct.

In a real RAG system this is the **indexing** step: every time documents are added to the knowledge base, they must be indexed (vectorized) again before they can be retrieved.

**Extra observation:** Document 0 (*"**Transformers** use self-attention..."*) still gets a score of **0** for *"transformer models"*, because TF-IDF treats *transformer* and *transformers* as two different words. This is the same limitation described in section 2: sparse retrieval matches words, not meanings. Dense retrieval with embeddings would recognise that the two documents are related.

In [9]:
# Both transformer documents are found only when both word forms are in the query
search_filtered("transformer transformers", k=3)

Score: 0.289 | BERT is a transformer model that uses bidirectional context.
Score: 0.273 | Transformers use self-attention to understand relationships between words.


__________________________________________________________________________________________

# What I Learned from Lab 13 (Part 2)

### A reusable search function
- `search(query, k)` puts the whole retrieval pipeline in one function: transform the query, compute cosine similarity, sort the scores and print the top-k documents.

### Rank is not the same as relevance
- `search()` always returns `k` documents, even for an unrelated query like *"volcano eruption"*, where every score is **0.000**.
- `search_filtered()` prints only documents with a **positive score** and prints *"No matching documents found."* otherwise. In RAG this is important: giving irrelevant documents to the language model can lead to wrong answers.

### Keeping the index up to date
- After adding the BERT document, the old vectors could not find it. Rebuilding with `fit_transform(documents)` updated the vocabulary (39 → 45 words), the IDF weights and the document vectors (6 → 7 rows), and then the BERT document was ranked first for *"transformer models"* (score ≈ 0.408).

### Sparse vs. dense retrieval
- **Sparse retrieval (TF-IDF)** is simple and fast, but it only matches **exact words**: *transformer ≠ transformers*, *automobiles ≠ cars*.
- **Dense retrieval** uses learned embeddings and can match **meanings**, even with different words, but it needs a pretrained embedding model.
- Modern RAG systems can use sparse retrieval, dense retrieval, or a combination of both (hybrid search).